# 04 - Design de PCBs
> Revisao de esquematicos, documentacao e calculos para placas de circuito

**Modulo:** `12_enfitec_servicos` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

## Revisao de Esquematicos

Revisao automatizada de esquematicos para identificar erros comuns
antes de enviar para fabricacao.

In [ ]:
system_pcb = (
    "You are a senior PCB design engineer with 20 years of experience. "
    "You review schematics for errors, missing components, and best practices. "
    "Use ASCII-only characters in all responses."
)

schematic_desc = """
Schematic description for review - ESP32 sensor board:

POWER SECTION:
- USB-C connector (power only) -> 5V rail
- AMS1117-3.3 LDO: Vin=5V, Vout=3.3V
- Input capacitor: 10uF electrolytic on Vin
- Output capacitor: 10uF electrolytic on Vout
- No reverse polarity protection

MCU SECTION:
- ESP32-WROOM-32 module
- EN pin: 10k pull-up to 3.3V
- GPIO0 (boot): floating (directly to header)
- No decoupling caps on 3.3V pin of ESP32
- Crystal: external 40MHz (no load caps shown)

SENSOR SECTION:
- ADS1115 ADC on I2C (SDA=GPIO21, SCL=GPIO22)
- No pull-up resistors on I2C lines
- ADDR pin floating
- Analog input directly from sensor (no protection)

COMMUNICATION:
- RS485 transceiver (MAX485): DE/RE tied together to GPIO4
- No termination resistor
- No ESD protection on bus lines

CONNECTORS:
- 4-pin screw terminal for sensor input
- 2-pin screw terminal for RS485 (A, B only, no GND)
- 10-pin debug header (JTAG)
"""

resp = ask(
    f"Review this schematic for errors and missing components. "
    f"Rate each issue as CRITICAL, WARNING, or SUGGESTION.\n\n"
    f"{schematic_desc}",
    system=system_pcb,
    model=SONNET,
    max_tokens=2048
)
print(resp)

In [ ]:
# Checklist automatizado de revisao
checklist = {
    "Capacitores de desacoplamento": {
        "regra": "100nF ceramico em cada pino de alimentacao do IC",
        "status": "FALHA - ESP32 sem decoupling caps"
    },
    "Pull-ups I2C": {
        "regra": "4.7k pull-up em SDA e SCL para 3.3V",
        "status": "FALHA - I2C sem pull-ups"
    },
    "Protecao ESD": {
        "regra": "TVS ou diodo de clamping em todas as interfaces externas",
        "status": "FALHA - RS485 e sensor sem protecao"
    },
    "Boot mode ESP32": {
        "regra": "GPIO0 com pull-up 10k + botao para GND",
        "status": "FALHA - GPIO0 flutuante"
    },
    "Protecao contra inversao": {
        "regra": "Diodo ou MOSFET P na entrada de alimentacao",
        "status": "FALHA - sem protecao de polaridade"
    },
    "Terminacao RS485": {
        "regra": "Resistor 120 ohm entre A e B (ultimo no do barramento)",
        "status": "FALHA - sem terminacao"
    },
    "GND no conector RS485": {
        "regra": "Pino de GND de referencia no conector",
        "status": "FALHA - apenas A e B"
    }
}

falhas = sum(1 for v in checklist.values() if "FALHA" in v["status"])
total = len(checklist)

print(f"=== RESULTADO DA REVISAO ===")
print(f"Itens verificados: {total}")
print(f"Falhas encontradas: {falhas}")
print(f"Taxa de aprovacao: {(total-falhas)/total*100:.0f}%\n")

for item, info in checklist.items():
    icon = "[FAIL]" if "FALHA" in info["status"] else "[ OK ]" 
    print(f"{icon} {item}")
    print(f"       {info['status']}")

## Calculo de Largura de Trilha

Calculo baseado na norma IPC-2221 para dimensionamento de trilhas de PCB.

In [ ]:
resp = ask(
    "Explain and calculate PCB trace width using the IPC-2221 standard for:\n"
    "- Current: 3A continuous\n"
    "- Copper thickness: 1 oz/ft2 (35um)\n"
    "- Maximum temperature rise: 10C\n"
    "- Trace is on external layer\n\n"
    "Show the complete IPC-2221 formula step by step:\n"
    "1. Area = (I / (k1 * dT^k2))^(1/k3)\n"
    "2. Width = Area / (thickness * 1.378)\n\n"
    "Also calculate for 1A, 2A, 5A, and 10A for comparison.\n"
    "Include safety margin recommendations.",
    system=system_pcb,
    model=SONNET,
    max_tokens=2048
)
print(resp)

In [ ]:
import math

def trace_width_ipc2221(current_A, temp_rise_C, copper_oz=1.0, external=True):
    """
    Calcula largura de trilha pela IPC-2221.
    
    Args:
        current_A: corrente em Amperes
        temp_rise_C: elevacao de temperatura permitida (C)
        copper_oz: espessura do cobre (oz/ft2)
        external: True para camada externa, False para interna
    
    Returns:
        largura em mm e mils
    """
    # Constantes IPC-2221
    if external:
        k1, k2, k3 = 0.048, 0.44, 0.725
    else:
        k1, k2, k3 = 0.024, 0.44, 0.725
    
    # Espessura do cobre em mils (1 oz = 1.378 mils)
    thickness_mils = copper_oz * 1.378
    
    # Area da secao transversal (mils^2)
    area = (current_A / (k1 * temp_rise_C**k2))**(1.0/k3)
    
    # Largura (mils)
    width_mils = area / thickness_mils
    width_mm = width_mils * 0.0254
    
    return width_mm, width_mils

# Tabela de larguras para diferentes correntes
print(f"{'Corrente':>10} {'Largura (mm)':>14} {'Largura (mils)':>16} {'Recomendado (mm)':>18}")
print("-" * 62)

for I in [0.5, 1.0, 2.0, 3.0, 5.0, 10.0]:
    w_mm, w_mils = trace_width_ipc2221(I, temp_rise_C=10, copper_oz=1.0)
    # Margem de seguranca de 50%
    w_rec = math.ceil(w_mm * 1.5 * 10) / 10  # arredonda para 0.1mm
    print(f"{I:>8.1f} A {w_mm:>12.3f} mm {w_mils:>14.1f} mil {w_rec:>16.1f} mm")

## Dissipacao Termica em PCBs

Calculos de dissipacao termica para garantir operacao dentro dos limites.

In [ ]:
resp = ask(
    "Calculate thermal dissipation for a PCB with these components:\n"
    "1. LDO regulator AMS1117-3.3: Vin=12V, Vout=3.3V, Iout=800mA\n"
    "2. ESP32 module: 250mW average (WiFi active)\n"
    "3. RS485 transceiver: 100mW\n"
    "4. ADS1115 ADC: 2mW\n\n"
    "For each component calculate:\n"
    "- Power dissipation (W)\n"
    "- Junction temperature (use typical thermal resistance values)\n"
    "- Whether a heat sink or thermal vias are needed\n\n"
    "PCB: FR4, 2 layers, 1oz copper, ambient temp 40C max.\n"
    "Provide recommendations for thermal management.",
    system=system_pcb,
    model=SONNET,
    max_tokens=2048
)
print(resp)

In [ ]:
# Calculos termicos
T_amb = 40  # temperatura ambiente maxima (C)

components = [
    {
        "nome": "AMS1117-3.3 (LDO)",
        "Vin": 12.0, "Vout": 3.3, "I": 0.8,
        "Rth_jc": 15,   # C/W (junction to case)
        "Rth_ca": 50,   # C/W (case to ambient, SOT-223 on PCB)
        "Tj_max": 125   # C
    },
    {
        "nome": "ESP32-WROOM-32",
        "P_fixed": 0.250,
        "Rth_jc": 10,
        "Rth_ca": 40,
        "Tj_max": 105
    },
    {
        "nome": "MAX485 (RS485)",
        "P_fixed": 0.100,
        "Rth_jc": 80,
        "Rth_ca": 100,
        "Tj_max": 150
    }
]

print(f"{'Componente':<25} {'Pdiss (W)':>10} {'Tj (C)':>8} {'Margem':>10} {'Status':>10}")
print("-" * 68)

total_power = 0
for c in components:
    # Potencia dissipada
    if "P_fixed" in c:
        P = c["P_fixed"]
    else:
        P = (c["Vin"] - c["Vout"]) * c["I"]
    
    total_power += P
    
    # Temperatura de juncao
    Rth_total = c["Rth_jc"] + c["Rth_ca"]
    Tj = T_amb + P * Rth_total
    
    # Margem
    margem = c["Tj_max"] - Tj
    status = "OK" if margem > 20 else "ATENCAO" if margem > 0 else "CRITICO"
    
    print(f"{c['nome']:<25} {P:>10.3f} {Tj:>8.1f} {margem:>8.1f} C {status:>10}")

print(f"\nPotencia total dissipada: {total_power:.3f} W")
print(f"\nNota: LDO com {(12-3.3)*0.8:.1f}W precisa de dissipacao termica adequada!")

## Gerando BOM (Bill of Materials)

Geracao automatizada de BOM estruturado a partir de descricao do projeto.

In [ ]:
resp = ask(
    "Generate a detailed Bill of Materials (BOM) for an ESP32 sensor board with:\n"
    "- ESP32-WROOM-32 module\n"
    "- AMS1117-3.3 voltage regulator\n"
    "- ADS1115 16-bit ADC\n"
    "- MAX485 RS485 transceiver\n"
    "- USB-C connector (power only)\n"
    "- All necessary passives (decoupling caps, pull-ups, etc.)\n"
    "- ESD protection (TVS diodes)\n"
    "- LED indicators (power, status, comms)\n"
    "- Screw terminals for sensor and RS485\n\n"
    "Format as a table with columns:\n"
    "Reference | Part Number | Description | Package | Qty | Supplier | Est. Cost (USD)\n\n"
    "Use real part numbers from LCSC/Digikey where possible.\n"
    "Include total estimated cost.",
    system=system_pcb,
    model=SONNET,
    max_tokens=2048
)
print(resp)

In [ ]:
# BOM estruturado em Python para exportacao
bom = [
    {"ref": "U1",  "pn": "ESP32-WROOM-32",  "desc": "WiFi/BT module",          "pkg": "Module",  "qty": 1, "cost": 3.50},
    {"ref": "U2",  "pn": "AMS1117-3.3",      "desc": "LDO regulator 3.3V",      "pkg": "SOT-223", "qty": 1, "cost": 0.15},
    {"ref": "U3",  "pn": "ADS1115IDGSR",     "desc": "16-bit ADC I2C",          "pkg": "MSOP-10", "qty": 1, "cost": 1.80},
    {"ref": "U4",  "pn": "MAX485ESA+T",      "desc": "RS485 transceiver",       "pkg": "SO-8",    "qty": 1, "cost": 0.80},
    {"ref": "U5",  "pn": "PESD5V0U2BT",     "desc": "ESD protection dual",     "pkg": "SOT-23",  "qty": 3, "cost": 0.10},
    {"ref": "C1-C6", "pn": "CL10B104KB8",   "desc": "100nF ceramic MLCC",      "pkg": "0402",    "qty": 6, "cost": 0.01},
    {"ref": "C7-C8", "pn": "CL21A106KOQ",   "desc": "10uF ceramic MLCC",       "pkg": "0805",    "qty": 2, "cost": 0.05},
    {"ref": "R1-R2", "pn": "RC0402FR-074K7", "desc": "4.7k I2C pull-up",        "pkg": "0402",    "qty": 2, "cost": 0.01},
    {"ref": "R3-R4", "pn": "RC0402FR-0710K", "desc": "10k pull-up/down",        "pkg": "0402",    "qty": 2, "cost": 0.01},
    {"ref": "R5",  "pn": "RC0402FR-07120R",  "desc": "120R RS485 term.",         "pkg": "0402",    "qty": 1, "cost": 0.01},
    {"ref": "R6-R8", "pn": "RC0402FR-07330R", "desc": "330R LED resistor",      "pkg": "0402",    "qty": 3, "cost": 0.01},
    {"ref": "D1-D3", "pn": "19-217/GHC",     "desc": "LED green/red/yellow",    "pkg": "0603",    "qty": 3, "cost": 0.03},
    {"ref": "J1",  "pn": "USB4110-GF-A",     "desc": "USB-C connector",         "pkg": "SMD",     "qty": 1, "cost": 0.40},
    {"ref": "J2",  "pn": "1729128",          "desc": "4-pin screw terminal",    "pkg": "5.08mm",  "qty": 1, "cost": 0.30},
    {"ref": "J3",  "pn": "1729128",          "desc": "3-pin screw terminal",    "pkg": "5.08mm",  "qty": 1, "cost": 0.25},
]

total = 0
print(f"{'Ref':<8} {'Part Number':<20} {'Descricao':<25} {'Pkg':<10} {'Qty':>4} {'Custo':>8}")
print("=" * 78)
for item in bom:
    line_cost = item["qty"] * item["cost"]
    total += line_cost
    print(f"{item['ref']:<8} {item['pn']:<20} {item['desc']:<25} {item['pkg']:<10} {item['qty']:>4} ${line_cost:>6.2f}")

print("=" * 78)
print(f"{'TOTAL':<68} ${total:>6.2f}")
print(f"\nCusto estimado por placa (componentes): ${total:.2f}")
print(f"Custo estimado com PCB (JLCPCB 5pcs): ${total + 2.00:.2f}")

## Design Rules para Fabricacao

Regras de design para fabricantes comuns e suas capacidades.

In [ ]:
resp = ask(
    "Provide a detailed comparison of PCB manufacturing design rules for:\n"
    "1. JLCPCB (standard and advanced)\n"
    "2. PCBWay (standard)\n"
    "3. OSHPARK\n\n"
    "For each, list:\n"
    "- Minimum trace width/spacing\n"
    "- Minimum via drill/annular ring\n"
    "- Solder mask clearance and minimum dam\n"
    "- Board thickness options\n"
    "- Copper weight options\n"
    "- Minimum hole size (PTH and NPTH)\n"
    "- Silkscreen minimum line width and font size\n\n"
    "Format as a comparison table.\n"
    "Also provide practical DFM tips for hobby/prototype boards.",
    system=system_pcb,
    model=SONNET,
    max_tokens=2048
)
print(resp)

In [ ]:
# Design rules como dicionario para validacao automatizada
design_rules = {
    "JLCPCB_standard": {
        "min_trace_mm": 0.127,          # 5 mil
        "min_space_mm": 0.127,          # 5 mil
        "min_via_drill_mm": 0.3,
        "min_via_annular_mm": 0.15,
        "min_hole_pth_mm": 0.3,
        "min_solder_mask_dam_mm": 0.1,
        "min_silkscreen_mm": 0.15,
        "copper_oz": [1, 2],
        "layers": [1, 2, 4, 6],
        "price_5pcs_usd": 2.0
    },
    "JLCPCB_advanced": {
        "min_trace_mm": 0.09,           # 3.5 mil
        "min_space_mm": 0.09,
        "min_via_drill_mm": 0.15,
        "min_via_annular_mm": 0.1,
        "min_hole_pth_mm": 0.15,
        "min_solder_mask_dam_mm": 0.08,
        "min_silkscreen_mm": 0.1,
        "copper_oz": [0.5, 1, 2, 3],
        "layers": [1, 2, 4, 6, 8, 10],
        "price_5pcs_usd": 15.0
    },
    "PCBWay_standard": {
        "min_trace_mm": 0.1,            # 4 mil
        "min_space_mm": 0.1,
        "min_via_drill_mm": 0.2,
        "min_via_annular_mm": 0.15,
        "min_hole_pth_mm": 0.2,
        "min_solder_mask_dam_mm": 0.1,
        "min_silkscreen_mm": 0.15,
        "copper_oz": [1, 2, 3],
        "layers": [1, 2, 4, 6, 8],
        "price_5pcs_usd": 5.0
    }
}

# Verificar se um design atende as regras
my_design = {
    "min_trace_mm": 0.2,
    "min_space_mm": 0.2,
    "min_via_drill_mm": 0.3,
    "min_via_annular_mm": 0.15
}

print("Verificacao de compatibilidade do design:\n")
for fab, rules in design_rules.items():
    ok = True
    for param, value in my_design.items():
        if param in rules and value < rules[param]:
            ok = False
            break
    status = "COMPATIVEL" if ok else "INCOMPATIVEL"
    print(f"  {fab:<25} -> {status}")

## Instrucoes de Montagem

Geracao de instrucoes passo a passo para montagem manual de PCBs.

In [ ]:
resp = ask(
    "Generate step-by-step assembly instructions for the ESP32 sensor board PCB.\n"
    "Components to assemble (from BOM):\n"
    "- SMD passives: 0402 resistors and capacitors\n"
    "- SMD ICs: AMS1117-3.3 (SOT-223), ADS1115 (MSOP-10), MAX485 (SO-8)\n"
    "- SMD LEDs: 0603\n"
    "- Module: ESP32-WROOM-32 (castellated pads)\n"
    "- THT: screw terminals, USB-C connector\n"
    "- ESD protection TVS diodes (SOT-23)\n\n"
    "Include:\n"
    "1. Recommended soldering order (smallest to largest)\n"
    "2. Temperature profile for each component type\n"
    "3. Required tools and consumables\n"
    "4. Test points to verify after each stage\n"
    "5. Common mistakes to avoid\n"
    "6. Final inspection checklist",
    system=system_pcb,
    model=SONNET,
    max_tokens=2048
)
print(resp)

In [ ]:
# Checklist de montagem interativo
assembly_steps = [
    {"etapa": 1, "desc": "Aplicar pasta de solda no stencil",
     "componentes": "N/A",
     "verificacao": "Pasta uniforme, sem excessos ou falhas"},
    {"etapa": 2, "desc": "Posicionar resistores e capacitores 0402",
     "componentes": "R1-R8, C1-C6",
     "verificacao": "Polaridade de capacitores (se aplicavel)"},
    {"etapa": 3, "desc": "Posicionar LEDs 0603",
     "componentes": "D1-D3",
     "verificacao": "Verificar polaridade (catodo marcado)"},
    {"etapa": 4, "desc": "Posicionar TVS diodes SOT-23",
     "componentes": "U5 (x3)",
     "verificacao": "Orientacao do pino 1"},
    {"etapa": 5, "desc": "Posicionar ICs SMD",
     "componentes": "U2 (AMS1117), U3 (ADS1115), U4 (MAX485)",
     "verificacao": "Pino 1 alinhado com marcacao da PCB"},
    {"etapa": 6, "desc": "Reflow no forno ou hot plate",
     "componentes": "Todos SMD",
     "verificacao": "Perfil: 150C preheat, 230C pico, <60s acima 220C"},
    {"etapa": 7, "desc": "Inspecao visual pos-reflow",
     "componentes": "Todos SMD",
     "verificacao": "Sem bridges, cold joints ou tombstones"},
    {"etapa": 8, "desc": "Teste de alimentacao (antes do ESP32)",
     "componentes": "N/A",
     "verificacao": "3.3V +/-50mV no pino de saida do LDO"},
    {"etapa": 9, "desc": "Soldar ESP32 module (reflow ou manual)",
     "componentes": "U1",
     "verificacao": "Todos os pads soldados, GND pad conectado"},
    {"etapa": 10, "desc": "Soldar conectores THT",
     "componentes": "J1 (USB-C), J2, J3 (screw terminals)",
     "verificacao": "Conectores firmes, solda nos dois lados"},
    {"etapa": 11, "desc": "Teste funcional completo",
     "componentes": "N/A",
     "verificacao": "Upload firmware, teste WiFi, leitura ADC, RS485"},
]

print("=== INSTRUCOES DE MONTAGEM - ESP32 Sensor Board ===")
print(f"Total de etapas: {len(assembly_steps)}\n")

for step in assembly_steps:
    print(f"Etapa {step['etapa']:>2}: {step['desc']}")
    print(f"          Componentes: {step['componentes']}")
    print(f"          Verificacao: {step['verificacao']}")
    print()

---

## Exercicios Propostos

1. **Revisao de esquematico** - Adicione ao esquematico descrito: protecao contra
   inversao de polaridade com MOSFET P, filtro EMI na entrada de alimentacao e
   peca ao Claude para revisar novamente.

2. **Calculo de impedancia** - Para um sinal de 100MHz em trilha microstrip,
   calcule a largura necessaria para impedancia de 50 ohm em FR4 (Er=4.5).

3. **BOM otimizado** - Peca ao Claude para otimizar o BOM substituindo componentes
   por alternativas mais baratas mantendo a funcionalidade.

4. **PCB 4 camadas** - Peca ao Claude para recomendar o stackup de uma PCB
   de 4 camadas para este projeto (sinais, GND, power, sinais).

5. **Gerber review** - Descreva os arquivos Gerber necessarios para fabricacao
   e peca ao Claude para criar um checklist de verificacao pre-fabricacao.

---

## Proximos Passos

- Explorar ferramentas de EDA open-source (KiCad) com automacao via Python
- Integrar revisao de esquematicos com CI/CD para projetos de hardware
- Estudar design para EMC (compatibilidade eletromagnetica)
- Implementar testes automatizados de PCB com fixtures de teste (bed of nails)